<a href="https://colab.research.google.com/github/PanditPranav/WildAlertModels_Circumstances/blob/main/notebooks/02_30092024_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from transformers import AutoTokenizer
from datasets import Dataset, load_dataset

from pathlib import Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
print(torch.cuda.is_available())
#torch.cuda.get_device_name(0)

True


In [4]:
#data_dir = Path("../data/interim/")
data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/processed")
raw_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/raw")
interim_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/interim")
ckpt = "bert-base-uncased"

## Tokenize text

In [5]:
tokenizer = AutoTokenizer.from_pretrained(ckpt, use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Exploring tokenizer and thoughts on improving model prediction with different tokenizers
1. Volcabulary is 305222,: larger vocab greater accuracy?
2. Is there a tokenizer that will be useful medical data
3.

In [6]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
df = pd.read_csv(interim_data_dir/"wildalert_circumstances_api_ohe.csv")
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.dropna(inplace=True)
df.head()

,text,terms,abduction_with_intent_of_rescue,animal_interaction,bicycle_collision,born_in_captivity,botanicals,bow_and_arrow,cat_interaction,collision,...,trapped_in_humane_/_cage_trap,trapped_in_leghold_/_trap_/_snare,tree_trimming,unauthorized_or_untrained_rehabilitation,undetermined,vehicle_collision,watercraft_collision,weather_event,wind_turbine_collision,window_/_wall_collision
0,"Mallard. In road, easy to catch, hit by car. ...",['vehicle_collision'],0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,Grey Fox. Orphan,['orphan'],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Malay Spotted Dove. Cat attack,['cat_interaction'],0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,"Mew Gull. Found on ground, in road, easy to ca...",['vehicle_collision'],0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,Common Merganser. Orphans,['orphan'],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
#df.columns.tolist()

In [9]:
df.shape

(36315, 73)

In [10]:
sample = df.text[:5]
sample[3]

'Mew Gull. Found on ground, in road, easy to catch - likely hit by car'

In [11]:
tokenizer(sample[3])

{'input_ids': [101, 2033, 2860, 19739, 3363, 1012, 2179, 2006, 2598, 1010, 1999, 2346, 1010, 3733, 2000, 4608, 1011, 3497, 2718, 2011, 2482, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
tokens = tokenizer.tokenize(sample[3])
tokens

['me',
 '##w',
 'gu',
 '##ll',
 '.',
 'found',
 'on',
 'ground',
 ',',
 'in',
 'road',
 ',',
 'easy',
 'to',
 'catch',
 '-',
 'likely',
 'hit',
 'by',
 'car']

In [13]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

[2033,
 2860,
 19739,
 3363,
 1012,
 2179,
 2006,
 2598,
 1010,
 1999,
 2346,
 1010,
 3733,
 2000,
 4608,
 1011,
 3497,
 2718,
 2011,
 2482]

In [14]:
tokenizer.decode(ids)

'mew gull. found on ground, in road, easy to catch - likely hit by car'

In [15]:
tokenizer.decode(tokenizer(sample[3])["input_ids"])

'[CLS] mew gull. found on ground, in road, easy to catch - likely hit by car [SEP]'

## Build the dataset

In [16]:
# Pick only the columns we need for the model
df = df[list(set(df.columns).difference({"patient_id", "terms"}))]
df.head()

,friendly,tree_trimming,physical_trauma,fire_/_smoke,weather_event,non-domestic_animal_interaction,plane_collision,entrapped_in_netting_/_string_/_wire,train_collision,bicycle_collision,...,powerline_/_wire_collision,entrapped_in_building,same_species_interaction,unauthorized_or_untrained_rehabilitation,tar_exposure,entrapped_in_fence,physical_trauma_by_unknown_cause,entrapped_in_water,displaced_from_nest,grease_exposure
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
#df.columns.tolist()

In [18]:
#df[['non-domestic_animal_interaction', 'non-domestic_animal_interaction',]]

In [19]:
# Split dataset into train, val & test set
_ds = (Dataset
          .from_pandas(df)
          .train_test_split(test_size=0.15,
                            shuffle=True)
     )
ds = _ds["train"].train_test_split(test_size=0.2)
ds["val"] = ds.pop("test")
ds["test"] = _ds["test"]

In [24]:
#df.columns.tolist()

In [25]:
#ds = ds.remove_columns(column_names=["__index_level_0__"])

In [26]:
labels = sorted(ds["train"].column_names)
labels.remove("text")
labels

['abduction_with_intent_of_rescue',
 'animal_interaction',
 'bicycle_collision',
 'born_in_captivity',
 'botanicals',
 'bow_and_arrow',
 'cat_interaction',
 'collision',
 'confiscation',
 'cooking_oil_exposure',
 'displaced_from_nest',
 'disturbed_metabolic_rest',
 'dog_interaction',
 'domestic_animal_interaction',
 'dumped',
 'electrocution',
 'entrapment',
 'entrapped_in_building',
 'entrapped_in_chimney',
 'entrapped_in_fence',
 'entrapped_in_fishing_tackle',
 'entrapped_in_litter_/_garbage',
 'entrapped_in_netting_/_string_/_wire',
 'entrapped_in_storm_drain_/_sewer',
 'entrapped_in_vehicle',
 'entrapped_in_water',
 'fire_/_smoke',
 'friendly',
 'garden_/_farm_equipment_collision',
 'gas_flare',
 'grease_exposure',
 'grounded',
 'gunshot',
 'hand_held_object_collision',
 'illness',
 'inappropriate_human_intervention',
 'maladaptation_/_failure_to_thrive',
 'mating_injury',
 'nest_/_habitat_disturbance_or_destruction',
 'non-domestic_animal_interaction',
 'non-weapon_projectile',
 '

In [27]:
# Add labels to the dataset as float (pytorch excepts float tensors)
ds = ds.map(lambda row: {"labels": [float(row[l]) for l in labels]})

Map:   0%|          | 0/24693 [00:00<?, ? examples/s]

Map:   0%|          | 0/6174 [00:00<?, ? examples/s]

Map:   0%|          | 0/5448 [00:00<?, ? examples/s]

In [28]:
ds["train"][0]

{'friendly': 0,
 'tree_trimming': 0,
 'physical_trauma': 0,
 'fire_/_smoke': 0,
 'weather_event': 0,
 'non-domestic_animal_interaction': 0,
 'plane_collision': 0,
 'entrapped_in_netting_/_string_/_wire': 0,
 'train_collision': 0,
 'bicycle_collision': 0,
 'watercraft_collision': 0,
 'window_/_wall_collision': 0,
 'nest_/_habitat_disturbance_or_destruction': 0,
 'bow_and_arrow': 0,
 'hand_held_object_collision': 0,
 'paint_exposure': 0,
 'domestic_animal_interaction': 0,
 'confiscation': 0,
 'solar_panel_collision': 0,
 'undetermined': 0,
 'entrapped_in_chimney': 0,
 'maladaptation_/_failure_to_thrive': 0,
 'entrapped_in_storm_drain_/_sewer': 0,
 'collision': 0,
 'dumped': 0,
 'nuisance_animal': 0,
 'abduction_with_intent_of_rescue': 0,
 'orphan': 1,
 'toxic_exposure': 0,
 'mating_injury': 0,
 'entrapped_in_fishing_tackle': 0,
 'text': 'Eastern Cottontail. found 3 days ago w/ lawnmower, unsure if mom returned',
 'gas_flare': 0,
 'dog_interaction': 0,
 'gunshot': 0,
 'illness': 0,
 'stra

In [29]:
ds["train"][226]

{'friendly': 0,
 'tree_trimming': 0,
 'physical_trauma': 0,
 'fire_/_smoke': 0,
 'weather_event': 0,
 'non-domestic_animal_interaction': 0,
 'plane_collision': 0,
 'entrapped_in_netting_/_string_/_wire': 0,
 'train_collision': 0,
 'bicycle_collision': 0,
 'watercraft_collision': 0,
 'window_/_wall_collision': 0,
 'nest_/_habitat_disturbance_or_destruction': 0,
 'bow_and_arrow': 0,
 'hand_held_object_collision': 0,
 'paint_exposure': 0,
 'domestic_animal_interaction': 0,
 'confiscation': 0,
 'solar_panel_collision': 0,
 'undetermined': 0,
 'entrapped_in_chimney': 0,
 'maladaptation_/_failure_to_thrive': 0,
 'entrapped_in_storm_drain_/_sewer': 1,
 'collision': 0,
 'dumped': 0,
 'nuisance_animal': 0,
 'abduction_with_intent_of_rescue': 0,
 'orphan': 0,
 'toxic_exposure': 0,
 'mating_injury': 0,
 'entrapped_in_fishing_tackle': 0,
 'text': 'Muscovy Duck. Orphaned, Collection from Wild, Orphan, Parents not available. 1 of 3 baby ducklings, Entrapped in storm drain. Rescued by Finder',
 'gas_

Let us look at how long is each of the text description

In [30]:
text_stats = np.array([len(row.split()) if row else 0 for row in ds["train"]["text"]])

In [31]:
#sns.histplot(text_stats, bins=20, kde=True)

In [32]:
(text_stats < 3).sum()/len(text_stats)

np.float64(0.010002834811485036)

1% of the data have words more than 128 words

In [33]:
(text_stats > 128).sum()/len(text_stats)

np.float64(0.0006074596039363382)

In [34]:
(text_stats > 200).sum()/len(text_stats)

np.float64(0.00012149192078726765)

In [35]:
for row in ds["train"]["text"]:
    if len(row.split()) < 3:
        print(row)

Mallard. Orphaned
LHSP. Grounded
Mallard. Botulism
MALL. sick
Mallard. Botulism
Mallard. illness
WTSH. Grounded
Mallard. Botulism
Mallard. Lost
Bushtit. injured
Mallard. Botulisme
Raccoon. orphan/transfer
Mallard. Lost
WTSH. Grounded
WTTR. Grounded
Raccoon. Electrocution
Killdeer. sick
ANHU. injured
WTSH. Grounded
WTSH. Grounded/DOA
NESH. Grounded
NESH. Grounded
Mallard. Botulism
Bluebird. burns
Bushtit. Trap
Mallard. botulism
Mallard. Botulism
ANHU. injured
Raccoon. Orphaned
Raccoon. Distemper
Mallard. Orphaned
NESH. Grounded
Starling. Grounded
NESH. Grounded
Mallard. botulisme?
PAGP. Collision/grounded
Kereru. Injured
Woodpigeon. Verzwakt
NESH. Grounded
Mallard. Botulism
WTSH. Grounded
Mallard. Botulisme
Kestrel. Grounded
Starling. CBC
Mallard. Botulisme
Mallard. Botulism
Mallard. Orphaned
Merlin. Grounded
ALHU. BIRDNAPPED
NESH. Grounded
Mallard. Botulisme
Cockatiel. Fire/smoke
Loggerhead. Beached
Mallard. Botulism
Mallard. Orphaned
Starling. Unk
Bushtit. Trap
WTSH. Grounded
Osprey. 

In [36]:
for row in ds["train"]["text"]:
    if len(row.split()) > 200:
        print(len(row.split()), row)
        print("-" * 20)

315 Black-crowned Night Heron. Reportedly the tree trimmers cut down the nesting tree or branches. Resort security gave them to Pam Ching. Pam Ching provided the following information: "I received these  two birds from the security at the Aulani Resort at Ko Olina on May 23.  I have never weighed them but they have surely gained rather than lost weight.  I feed them four times per day, around 16-18 smelt which I have defrosted and cut in half mixed with about 1/4 cup of Hills Ideal Balance dry cat food which has been soaked in water to soften it. This mixture (smelt and softened cat food) is mixed with 1/2 cup of water, poured into a Pyrex baking dish and placed in the cage.   I started out hand feeding the smelt but the birds quickly learned to eat on their own. Twice I have substituted the smelt with raw boneless, skinless chicken thighs which have been cut into bite sized pieces and extra fat trimmed off.
I have kept the birds outside  in a cage with absorbent bedding. The top and s

In [37]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [38]:
#tok_ds = ds.map(tokenizer, batched=True, num_proc=16, remove_columns=labels + ["text"])

In [39]:
#from huggingface_hub import login
#login(token=None)


In [40]:
%%time
def tok_fn(row):
  from transformers import AutoTokenizer
  ckpt = "bert-base-uncased"
  tokenizer = AutoTokenizer.from_pretrained(ckpt)
  return tokenizer(row["text"],
                  truncation=True,
                    padding="max_length",
                    max_length=128)


#def tok_fn(row):
#    return tokenizer(row["text"],
#                     truncation=True,
#                     padding="max_length",
#                     max_length=128)

tok_ds = ds.map(tok_fn, batched=True, num_proc=16, remove_columns=labels + ["text"])

Map (num_proc=16):   0%|          | 0/24693 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/6174 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5448 [00:00<?, ? examples/s]

CPU times: user 3.47 s, sys: 1.04 s, total: 4.51 s
Wall time: 11.9 s


In [41]:
tok_ds

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24693
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6174
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5448
    })
})

In [42]:
tok_ds["train"][0]

{'labels': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'input_ids': [101,
  2789,
  6557,
  14162,
  1012,
  2179,
  1017,
  2420,
  3283,
  1059,
  1013,
  10168,
  5302,
  13777,
  1010,
  12422,
  2065,
  3566,
  2513,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

In [44]:
# Save the processed data in a parquet file
for split,split_ds in tok_ds.items():
  output_path = data_dir / f"wildalert_circumstances_data_{split}.parquet"
  split_ds.to_parquet(output_path)

Creating parquet from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

In [46]:
%%time
data_files = {
    "train": str(data_dir / "wildalert_circumstances_data_train.parquet"),
    "val": str(data_dir /"wildalert_circumstances_data_val.parquet"),
    "test": str(data_dir /"wildalert_circumstances_data_test.parquet"),
}

ds = load_dataset("parquet", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

CPU times: user 306 ms, sys: 55.8 ms, total: 362 ms
Wall time: 945 ms
